In [0]:
from pyspark.sql.functions import (
    col,
    trim
)
from pyspark.sql.types import StringType
import pyspark.sql.functions as F

##### Importing from utils and validation_utils package

In [0]:
from utils.custom_utils import Transformations
from validation_utils.test_utils import Validations

tr_obj = Transformations()
va_obj = Validations()

#### Reading bronze table

In [0]:
df = spark.table("lakehouse.bronze.loc_a101")

### Silver layer transformations

#### 1. Renaming columns

In [0]:
rename_config = {
    "CID": "customer_number",
    "CNTRY": "country"
}
for old_name, new_name in rename_config.items():
    df = df.withColumnRenamed(old_name, new_name)

#### 2. Modifying customer_number
- Replacing '-' in customer_number with "" so that it can be used as joining key

In [0]:
df = df.withColumn(
    "customer_number",
    F.regexp_replace(col("customer_number"), "-", "")
)

#### 3. Removing records with no customer_number
- This table doesnot have any nulls in the customer_number

In [0]:
va_obj.null_check(df = df, primary_col = "customer_number")

#### 4. Removing customer_number duplicates
- This table doesnot have any customer_number duplicates

#### 5. Trimming

In [0]:
df = tr_obj.trim_columns(df)

#### 6. Cleaning country
- It has abbreviated values as well as non abbreviated values of country
- So, first identify the unique country values and normalize the abbreviated values 

In [0]:
df = df.withColumn(
    "country",
    F.when(col("country") == "DE", "Germany")
    .when(col("country").isin("US", "USA"), "United States")
    .when((col("country") == "") | (col("country").isNull()), "n/a")
    .otherwise(col("country"))
)

#### 7. Adding ingestion_ts column

In [0]:
df = tr_obj.add_ingestion_timestamp(df)

#### Dataframe sanity check

In [0]:
df.limit(10).display()

#### 8. Applying upsert(SCD type 1)

In [0]:
target_table = "lakehouse.silver.erp_customer_location"
if spark.catalog.tableExists(target_table):
    tr_obj.upsert(
        spark = spark,
        df = df,
        key_cols = ["customer_number"],
        table = "erp_customer_location",
        cdc = "ingestion_ts"
    )
else:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

#### Silver table sanity check

In [0]:
%sql
SELECT *
FROM lakehouse.silver.erp_customer_location
limit 10